In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
root_loc = '/Volumes/workspace/default/vanguard_data'

schema = StructType([
    StructField("Account Number", StringType(), False),
    StructField("Trade Date",StringType(), True),
    StructField("Settlement Date", StringType(), True),
    StructField("Transaction Type", StringType(), True),
    StructField("Transaction Description", StringType(), True),
    StructField("Investment Name", StringType(), True),
    StructField("Symbol", StringType(), True),
    StructField("Shares", DoubleType(), True),
    StructField("Share Price", DoubleType(), True),
    StructField("Principal Amount", DoubleType(), True),
    StructField("Commissions and Fees", DoubleType(), True),
    StructField("Net Amount", DoubleType(), True),
    StructField("Accrued Interest", DoubleType(), True),
    StructField("Account Type", StringType(), True)
    ])

In [0]:
df = (
  spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(schema)
    .load(root_loc)
)
df_clean = df.selectExpr(
    "`Account Number` as Account_Number",
    "`Trade Date` as Trade_Date",
    "`Settlement Date` as Settlement_Date",
    "`Transaction Type` as Transaction_Type",
    "`Transaction Description` as Transaction_Description",
    "`Investment Name` as Investment_Name",
    "Symbol",
    "Shares",
    "`Share Price` as Share_Price",
    "`Principal Amount` as Principal_Amount",
    "`Commissions and Fees` as Commissions_and_Fees",
    "`Net Amount` as Net_Amount",
    "`Accrued Interest` as Accrued_Interest",
    "`Account Type` as Account_Type",
    "_rescued_data"
)

In [0]:
(
  df_clean.writeStream
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/vanguard_data_checkpoint")
    .trigger(availableNow=True)
    .toTable("workspace.default.vanguard_raw_data")
)